## Homework 2: BERT on AWS

### 1. Setup and Import

Import required packages.

In [1]:
import os
import zipfile
import urllib.request
from io import BytesIO
import random
import json

import numpy as np
from numpy.linalg import norm
import pandas as pd

import boto3
from botocore.exceptions import ClientError

import torch
from transformers import AutoTokenizer, AutoModel
import pickle
from tqdm import tqdm

/home/ec2-user/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Define global variables.

In [2]:
BUCKET_NAME = "nky8459-de300-hw2"

RAW_DATA_PREFIX = "raw"
EMBEDDINGS_PREFIX = "embeddings"
OUTPUT_PREFIX = "outputs"

PRE1980_EMBED_KEY = f"{EMBEDDINGS_PREFIX}/movies_pre1980_bert-base-uncased.pkl"
FULL_EMBED_KEY = f"{EMBEDDINGS_PREFIX}/movies_full_bert-base-uncased.pkl"

PRE1980_COLD_KEY = f"{OUTPUT_PREFIX}/pre1980_cold_user.json"
PRE1980_TOP_KEY = f"{OUTPUT_PREFIX}/pre1980_top_user.json"

FULL_COLD_KEY = f"{OUTPUT_PREFIX}/full_cold_user.json"
FULL_TOP_KEY = f"{OUTPUT_PREFIX}/full_top_user.json"

MY_PROFILE_DAT_KEY = f"{RAW_DATA_PREFIX}/my_ratings.dat"
MY_RECS_KEY = f"{OUTPUT_PREFIX}/my_recommendations.json"

LOCAL_DATA_DIR = "/tmp/movielens_1m"

FILES = ["movies.dat", "ratings.dat", "users.dat"]
MODEL_NAME = "bert-base-uncased"

MOVIELENS_URL = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"

Create a S3 session for storing datasets.

In [3]:
def s3_setup():
    """Set up a S3 session."""
    
    session = boto3.Session()
    s3 = session.client("s3")
    return s3

In [4]:
s3 = s3_setup()

### Download Dataset

Helper functions:

In [5]:
def s3_object_exists(bucket, key):
    """Check if an object exists in the S3 bucket."""
    
    try:
        s3.head_object(Bucket=bucket, Key=key)
        return True
    except ClientError:
        return False

In [6]:
def download_and_extract_movielens():
    """Download the Movielens 1M dataset from url given, and extract the .zip folder."""
    
    os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
    zip_path = f"{LOCAL_DATA_DIR}/ml-1m.zip"

    print("Downloading MovieLens 1M...")
    urllib.request.urlretrieve(MOVIELENS_URL, zip_path)

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(LOCAL_DATA_DIR)

    print("Extraction complete.")

In [7]:
def upload_dataset_to_s3():
    """Upload the local downloaded .dat files to S3 bucket."""
    
    base_path = f"{LOCAL_DATA_DIR}/ml-1m"

    for filename in FILES:
        local_path = f"{base_path}/{filename}"
        print(f"Uploading {filename}...")
        s3.upload_file(local_path, BUCKET_NAME, f"{RAW_DATA_PREFIX}/{filename}")

In [8]:
def load_dat_from_s3(key, columns):
    """Load a .dat file in the S3 bucket. Convert it to a pandas dataframe and name the columns."""
    
    print(f"Loading {key} from S3...")
    obj = s3.get_object(Bucket=BUCKET_NAME, Key=key)
    content = obj["Body"].read()
    return pd.read_csv(
        BytesIO(content),
        sep="::",
        engine="python",
        names=columns,
        encoding="latin-1"
    )

Download the Movielens datasets if they don't already exist in S3.

In [9]:
def download():
    """Download the Movielens datasets if they don't exist in S3. Then load and return them."""
    
    # Check if dataset already exists in S3
    dataset_exists = all([s3_object_exists(BUCKET_NAME, f"{RAW_DATA_PREFIX}/{filename}") for filename in FILES])

    # Download and upload to S3 if not
    if dataset_exists:
        print("Dataset already exists in S3.")
    else:
        print("Dataset not found in S3.")
        download_and_extract_movielens()
        upload_dataset_to_s3()
        
    # Load datasets
    movies_df = load_dat_from_s3(
        f"{RAW_DATA_PREFIX}/movies.dat",
        ["movie_id", "title", "genres"]
    )
    ratings_df = load_dat_from_s3(
        f"{RAW_DATA_PREFIX}/ratings.dat",
        ["user_id", "movie_id", "rating", "timestamp"]
    )
    users_df = load_dat_from_s3(
        f"{RAW_DATA_PREFIX}/users.dat",
        ["user_id", "gender", "age", "occupation", "zipcode"]
    )
    
    return movies_df, ratings_df, users_df

In [10]:
movies_df, ratings_df, users_df = download()

Dataset already exists in S3.
Loading raw/movies.dat from S3...
Loading raw/ratings.dat from S3...
Loading raw/users.dat from S3...


### Pre-1980 Filtering

Inspect the movies dataframe:

In [11]:
movies_df.head()

,movie_id,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


The year of release is included in the title column. Extract that information as a new column. Then filter both movies and ratings and store them as new datasets.

In [12]:
def filter():
    """Filter the movies and ratings dataframe to only include movies released before 1980."""
    
    # Use regex expression to extract the release year from title
    movies_df["year"] = (
        movies_df["title"]
        .str.extract(r"\((\d{4})\)")
        .astype(int)
    )
    
    # Store the filtered movies as a new dataframe
    movies_pre_1980 = movies_df[movies_df["year"] <= 1980]
    
    # Match the filtered movies to ratings, then filter ratings as well
    pre_1980_movie_ids = set(movies_pre_1980["movie_id"])

    ratings_pre_1980 = ratings_df[
        ratings_df["movie_id"].isin(pre_1980_movie_ids)
    ]
    
    return movies_pre_1980, ratings_pre_1980

In [13]:
movies_pre_1980, ratings_pre_1980 = filter()

### 3. Embeddings

Each movie should be represented as a fixed-length vector so that semantically similar movies are close in vector space.

BERT initialization will only need to run once. This function uses the model 'bert-base-uncased', the standard and efficient baseline BERT model.

In [14]:
def initialize_bert():
    """Initialize a 'bert-base-uncased' model."""
    
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModel.from_pretrained(MODEL_NAME)
    model.eval()
    
    return tokenizer, model

In [15]:
tokenizer, model = initialize_bert()

/home/ec2-user/anaconda3/lib/python3.11/site-packages/transformers/utils/generic.py:260: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(


Embed movie title and genres, where the movie title also includes year of release in parenthesis.
- Include genres because titles alone can be ambiguous. This will help with clustering and similarity comparisons.

To convert the BERT output into one vector per movie, use mean pooling.
- Since movie titles vary in length, mean pooling accounts for this.
- It can capture the overall meaning across tokens, and is very interpretable.

In [16]:
def embed_text(text):
    """Embed given text with mean pooling."""
    
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )
    with torch.no_grad():
        outputs = model(**inputs)

    return outputs.last_hidden_state.mean(dim=1).squeeze().numpy()

In [17]:
def create_movie_embeddings(df):
    """Create embeddings in the format {title} {genres} for each movie."""
    
    embeddings = {}
    for _, row in tqdm(df.iterrows(), total=len(df)):
        text = f"{row['title']} {row['genres']}"
        embeddings[row["movie_id"]] = embed_text(text)
    return embeddings

Because BERT embeddings are expensive and reusable, only compute them once and upload to S3. Then always perform a check to look for stored embeddings in the bucket first, if it exists simply load from S3.

In [18]:
def embeddings(key, df):
    """Look for embeddings with the given bucket key. If found load it from S3.
    If not found create embeddings with the given dataframe."""
    
    if s3_object_exists(BUCKET_NAME, key):
        print("Embeddings found in S3. Loading...")

        obj = s3.get_object(
            Bucket=BUCKET_NAME,
            Key=key
        )
        movie_embeddings = pickle.loads(obj["Body"].read())

    else:
        print("Embeddings not found. Creating embeddings...")

        movie_embeddings = create_movie_embeddings(df)

        print("Uploading embeddings to S3...")
        s3.put_object(
            Bucket=BUCKET_NAME,
            Key=key,
            Body=pickle.dumps(movie_embeddings)
        )
    return movie_embeddings

In [19]:
movie_embeddings_pre1980 = embeddings(PRE1980_EMBED_KEY, movies_pre_1980)

Embeddings found in S3. Loading...


### 4. Recommendations

Generate movie recommendations from previously computed movie embeddings for two different user types:
- Cold user: A user that the system has no data on.
- Top user: A random user who has frequently rated movies (number of interactions among the top 5% of users).

**Cold User**

Since a cold user has no historical data and no embeddings to be constructed, recommendations are based on popularity of movies from the ratings.
- Use both the average ratings and the log of rating counts to find most-watched and well-rated movies.

In [20]:
def calculate_popularity(df):
    """Find the popularity score of movies in a given ratings dataframe."""
    
    # Aggregate popularity statistics
    movie_stats = (
        df
        .groupby("movie_id")
        .agg(
            rating_count=("rating", "count"),
            avg_rating=("rating", "mean")
        )
        .reset_index()
    )
    
    # Define a popularity score
    movie_stats["popularity_score"] = movie_stats["avg_rating"] * np.log1p(movie_stats["rating_count"])
    
    return movie_stats

Recommend the top 5 pre-1980 movies for cold users based on popularity scores.
- Last interaction time is None because the cold user has no rating history.
- User summary contains useful information such as
    - User id: None for cold users as they don't have any data.
    - Number of ratings: 0.
    - Movie release time: This specifies which movies dataframe we're using (pre-1980 in this case)

In [21]:
def recommend_top_5_cold(release_time, ratings, movies):
    """Recommend the top 5 movies based on popularity for cold users. Output all information as the required format in a dictionary."""
    
    recs = (
        calculate_popularity(ratings)
        .sort_values("popularity_score", ascending=False)
        .head(5)
    )

    cold_recommendations = (
        recs
        .merge(movies, on="movie_id")
        [["movie_id", "title"]]
    )
    
    output = {
        "User_Type": "Cold User",
        "Last_Interaction_Time": None,
        "User_Summary": {
            "user_id": None,
            "num_ratings": 0,
            "movie_release_time": release_time,
        },
        "Recommended_Movies": cold_recommendations.to_dict(orient="records")
    }
    
    return output

**Top User**

For this type of user, a content-based recommendation is possible by constructing a user embedding as a weighted average of movie embeddings.

First identify top users:

In [22]:
def identify_top_users(df):
    """Identify users who frequently rated movies and mark them as top users.
    Return a random top user and their ratings in the given dataframe."""
    
    ratings_per_user = df.groupby("user_id").size()

    threshold = ratings_per_user.quantile(0.95)
    top_user_ids = ratings_per_user[ratings_per_user >= threshold].index.tolist()

    top_user_id = random.choice(top_user_ids)
    
    top_user_ratings = df[
        df["user_id"] == top_user_id
    ]
    
    return top_user_id, top_user_ratings

Then build the user embedding with the weighted average of movie embeddings given a ratings dataframe:

In [23]:
def build_user_embedding(df, movie_embeddings):
    """Build user embedding via weight average of movie embeddings,
    with the given ratings dataframe."""
    
    vectors = []
    weights = []

    for _, row in df.iterrows():
        movie_id = row["movie_id"]
        if movie_id in movie_embeddings:
            vectors.append(movie_embeddings[movie_id])
            weights.append(row["rating"])

    return np.average(vectors, axis=0, weights=weights)

Compute similarily scores via cosine similarities.

In [24]:
def similarity_scores(top_user_ratings, top_user_embedding, movie_embeddings):
    """Calculate cosine similaritiy scores for the unseen movies of a user."""

    seen_movies = set(top_user_ratings["movie_id"])

    scores = []
    for movie_id, emb in movie_embeddings.items():
        if movie_id not in seen_movies:
            # Cosine similarity
            score = np.dot(top_user_embedding, emb) / (norm(top_user_embedding) * norm(emb))
            scores.append((movie_id, score))
    
    return scores

Recommend the top 5 movies for a top user (or a user that has data). Record the last interaction time as the maximum timestamp in the user's ratings, and output the same fields as before (in the same format for cold users).

In [25]:
def recommend_top_5_top(release_time, scores, top_user_ratings, top_user_id, movies, user_type):
    """Recommend the top 5 movies for a top user (or a user that has data) with 
    the given movie release time (pre-1980 or full dataset). Return the output as a dictionary."""
    
    top_movie_ids = [
        movie_id for movie_id, _ in
        sorted(scores, key=lambda x: x[1], reverse=True)[:5]
    ]

    top_user_recommendations = (
        movies[movies["movie_id"].isin(top_movie_ids)]
        [["movie_id", "title"]]
    )
    
    last_interaction_time = top_user_ratings["timestamp"].max() if user_type == "Top User" else 0
    
    top_user_output = {
        "User_Type": user_type,
        "Last_Interaction_Time": int(last_interaction_time),
        "User_Summary": {
            "user_id": top_user_id,
            "num_ratings": int(len(top_user_ratings)),
            "movie_release_time": release_time,
        },
        "Recommended_Movies": top_user_recommendations.to_dict(orient="records")
    }
    
    return top_user_output

**Save & Upload to S3**

Putting the outputs together in a function and save to S3 bucket as json files:

In [26]:
def save_json_to_s3(obj, bucket, key):
    """Save an object to the given bucket key."""
    
    s3.put_object(
        Bucket=bucket,
        Key=key,
        Body=json.dumps(obj, indent=2).encode("utf-8"),
        ContentType="application/json"
    )
    print(f"Saved to {key}")

In [27]:
def recommend(release_time, ratings, movies, embeddings, cold_key, top_key):
    """Recommend the top 5 movies for a cold user and a top user with a given
    dataframe (either pre-1980 or full dataset)."""
    
    # Cold users
    cold_output = recommend_top_5_cold(release_time, ratings, movies)
    
    # Top users
    top_user_id, top_user_ratings = identify_top_users(ratings)
    top_user_embedding = build_user_embedding(top_user_ratings, embeddings)
    scores = similarity_scores(top_user_ratings, top_user_embedding, embeddings)
    
    top_output = recommend_top_5_top(release_time, scores, top_user_ratings, top_user_id, movies, "Top User")
    
    # Save to S3
    save_json_to_s3(cold_output, BUCKET_NAME, cold_key)
    save_json_to_s3(top_output, BUCKET_NAME, top_key)
    
    return cold_output, top_output

In [28]:
pre_cold_output, pre_top_output = recommend("pre-1980", ratings_pre_1980, movies_pre_1980, movie_embeddings_pre1980, PRE1980_COLD_KEY, PRE1980_TOP_KEY)

Saved to outputs/pre1980_cold_user.json
Saved to outputs/pre1980_top_user.json


Example outputs:

In [29]:
pre_cold_output

{'User_Type': 'Cold User',
 'Last_Interaction_Time': None,
 'User_Summary': {'user_id': None,
  'num_ratings': 0,
  'movie_release_time': 'pre-1980'},
 'Recommended_Movies': [{'movie_id': 260,
   'title': 'Star Wars: Episode IV - A New Hope (1977)'},
  {'movie_id': 858, 'title': 'Godfather, The (1972)'},
  {'movie_id': 1196,
   'title': 'Star Wars: Episode V - The Empire Strikes Back (1980)'},
  {'movie_id': 912, 'title': 'Casablanca (1942)'},
  {'movie_id': 1193, 'title': "One Flew Over the Cuckoo's Nest (1975)"}]}

In [30]:
pre_top_output

{'User_Type': 'Top User',
 'Last_Interaction_Time': 975994851,
 'User_Summary': {'user_id': 984,
  'num_ratings': 272,
  'movie_release_time': 'pre-1980'},
 'Recommended_Movies': [{'movie_id': 970, 'title': 'Beat the Devil (1954)'},
  {'movie_id': 3229, 'title': "Another Man's Poison (1952)"},
  {'movie_id': 3405, 'title': 'Night to Remember, A (1958)'},
  {'movie_id': 3627, 'title': 'Carnival of Souls (1962)'},
  {'movie_id': 3741, 'title': 'Badlands (1973)'}]}

### 5. Repeat with Full Dataset

Reuse the same pipeline but for a different dataset. First create embeddings:

In [31]:
full_movie_embeddings = embeddings(FULL_EMBED_KEY, movies_df)

Embeddings found in S3. Loading...


Then build the recommendations:

In [32]:
full_cold_output, full_top_output = recommend("full range", ratings_df, movies_df, full_movie_embeddings, FULL_COLD_KEY, FULL_TOP_KEY)

Saved to outputs/full_cold_user.json
Saved to outputs/full_top_user.json


In [33]:
full_cold_output

{'User_Type': 'Cold User',
 'Last_Interaction_Time': None,
 'User_Summary': {'user_id': None,
  'num_ratings': 0,
  'movie_release_time': 'full range'},
 'Recommended_Movies': [{'movie_id': 260,
   'title': 'Star Wars: Episode IV - A New Hope (1977)'},
  {'movie_id': 2858, 'title': 'American Beauty (1999)'},
  {'movie_id': 318, 'title': 'Shawshank Redemption, The (1994)'},
  {'movie_id': 1198, 'title': 'Raiders of the Lost Ark (1981)'},
  {'movie_id': 527, 'title': "Schindler's List (1993)"}]}

In [34]:
full_top_output

{'User_Type': 'Top User',
 'Last_Interaction_Time': 1045841716,
 'User_Summary': {'user_id': 4312,
  'num_ratings': 883,
  'movie_release_time': 'full range'},
 'Recommended_Movies': [{'movie_id': 93,
   'title': 'Vampire in Brooklyn (1995)'},
  {'movie_id': 1636, 'title': 'Stag (1997)'},
  {'movie_id': 2523, 'title': 'Rollercoaster (1977)'},
  {'movie_id': 3107, 'title': 'Backdraft (1991)'},
  {'movie_id': 3476, 'title': "Jacob's Ladder (1990)"}]}

### 6. User Profile

The following 10 movies are rated to construct a personal user profile:

In [35]:
MY_RATINGS = {
    "Star Wars: Episode IV - A New Hope (1977)": 5,
    "Godfather, The (1972)": 5,
    "Raiders of the Lost Ark (1981)": 4,
    "Caught (1996)": 5,
    "Boiling Point (1993)": 5,
    "American Beauty (1999)": 4,
    "Forrest Gump (1994)": 4,
    "Shawshank Redemption, The (1994)": 5,
    "Toy Story (1995)": 4,
    "Dog Park (1998)": 5,
}

Then the movie IDs need to be matched to those titles. Construct a ratings dataframe based on the above movies and ratings, then convert to .dat format and upload it to S3 bucket.
- Use the made-up user id "me". This isn't important in recommendation algorithm.
- Won't have a timestamp column because it's also not used in the recommendation system.

In [36]:
def personal_ratings():
    """Find corresponding movie IDs of the rated movies, then construct a ratings dataframe."""
    
    title_to_id = dict(zip(movies_df["title"], movies_df["movie_id"]))
    
    my_profile_rows = []

    for title, rating in MY_RATINGS.items():
        if title not in title_to_id:
            raise ValueError(f"Movie not found in dataset: {title}")

        my_profile_rows.append({
            "user_id": "me",
            "movie_id": int(title_to_id[title]),
            "rating": rating
        })
    
    return pd.DataFrame(my_profile_rows)

In [40]:
def upload_personal_ratings(my_ratings_df):
    """Save the ratings dataframe to S3 bucket in .dat format."""
    
    dat_body = "\n".join(
        f"me::{row.movie_id}::{row.rating}"
        for _, row in my_ratings_df.iterrows()
    )
    
    print("Uploading personal ratings data to S3...")
    s3.put_object(
        Bucket=BUCKET_NAME,
        Key=MY_PROFILE_DAT_KEY,
        Body=dat_body.encode("utf-8")
    )

Build the personal user profile, using identical logic to the top users. The output will look similar to a top user's recommendation except:
- 0 for last interaction time, as the personal profile doesn't have a valid interaction.
- "me" for user id.

Then save this recommendation to S3.

In [41]:
def personal_profile():
    
    my_ratings_df = personal_ratings()
    upload_personal_ratings(my_ratings_df)
    
    my_user_embedding = build_user_embedding(my_ratings_df, full_movie_embeddings)
    scores = similarity_scores(my_ratings_df, my_user_embedding, full_movie_embeddings)
    my_recommendations_output = recommend_top_5_top("full range", scores, my_ratings_df, "me", movies_df, "Personal")
    
    save_json_to_s3(
        my_recommendations_output,
        BUCKET_NAME,
        MY_RECS_KEY
    )
    
    return my_recommendations_output

In [42]:
my_recommendations_output = personal_profile()
my_recommendations_output

Uploading personal ratings data to S3...
Saved to outputs/my_recommendations.json


{'User_Type': 'Personal',
 'Last_Interaction_Time': 0,
 'User_Summary': {'user_id': 'me',
  'num_ratings': 10,
  'movie_release_time': 'full range'},
 'Recommended_Movies': [{'movie_id': 42, 'title': 'Dead Presidents (1995)'},
  {'movie_id': 51, 'title': 'Guardian Angel (1994)'},
  {'movie_id': 517, 'title': 'Rising Sun (1993)'},
  {'movie_id': 1667, 'title': 'Mad City (1997)'},
  {'movie_id': 2942, 'title': 'Flashdance (1983)'}]}